---
## Step 0 — Environment Setup

Run this once to clone the repo and install all dependencies.

# CT-CLIP — Full Inference Pipeline with `CT_LiPro_v2.pt` (ClassFine)

**Model:** `CT_LiPro_v2.pt` — the **ClassFine** fine-tuned variant of CT-CLIP  
**Task:** Multi-abnormality detection across **18 chest CT pathologies**  
**Inference time:** ~0.5 s per volume  
**Paper:** *Generalist Foundation Models from a Multimodal Dataset for 3D Computed Tomography* (Hamamci et al., 2024) — [arXiv:2403.17834](https://arxiv.org/abs/2403.17834)  
**Repo:** https://github.com/ibrahimethemhamamci/CT-CLIP

---

## Pipeline Overview

```
NIfTI CT Volume (.nii.gz)
        │
        ▼
  [1] Preprocessing
      • HU Windowing (W=1500, L=-600)
      • Resample → isotropic voxels
      • Resize  → (240 × 480 × 480)
      • Normalize → [0, 1]
        │
        ▼
  [2] CT-ViT Vision Encoder
      (spatial + causal transformer)
        │
        ▼
  [3] ClassFine Head (CT_LiPro_v2)
      Linear(512 → 18)  +  Sigmoid
        │
        ▼
  [4] Probabilities for 18 Pathologies
        │
        ▼
  [5] Visualisation & Report
```

## Available Pretrained Weights

| File | Variant | Inference | Capability |
|------|---------|-----------|------------|
| `CT-CLIP_v2.pt` | Zero-shot | ~1.5 s | Open-vocabulary, retrieval |
| `CT_VocabFine_v2.pt` | VocabFine | ~1.5 s | Open-vocabulary, retrieval |
| **`CT_LiPro_v2.pt`** | **ClassFine ← this notebook** | **~0.5 s** | **18-class detection** |
| `RadBertClassifier.pth` | Text Classifier | — | Report label extraction |


In [9]:
# ── Clone repository ────────────────────────────────────────────────────────
import os, sys

REPO_DIR = "CT-CLIP"  # change if you want a different location

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ibrahimethemhamamci/CT-CLIP.git
else:
    print(f"Repo already cloned at ./{REPO_DIR}")

# ── Install packages into CURRENT notebook kernel ───────────────────────────
# Use %pip (not !pip) so installs go to this exact Jupyter kernel environment.
%pip install -q -e CT-CLIP/transformer_maskgit
%pip install -q -e CT-CLIP/CT_CLIP
%pip install -q nibabel SimpleITK scikit-image scipy matplotlib seaborn plotly huggingface_hub tqdm pandas

print("\n✅ Installation complete for current kernel.")
print(f"Python executable: {sys.executable}")

Repo already cloned at ./CT-CLIP
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [10 lines of output]
      ERROR: Could not find a version that satisfies the requirement setuptools>=40.8.0 (from versions: none)
      ERROR: No matching distribution found for setuptools>=40.8.0
      
      [notice] A new release of pip is available: 24.0 -> 26.1.1
      [notice] To update, run: python.exe -m pip install --upgrade pip
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [10 lines of output]
      ERROR: Could not find a version that satisfies the requirement setuptools>=40.8.0 (from versions: none)
      ERROR: No matching distribution found for setuptools>=40.8.0
      
      [notice] A new release of pip is available: 24.0 -> 26.1.1
      [notice] To update, run: python.exe -m pip install --upgrade pip
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

✅ Installation complete for current kernel.
Python executable: c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Scripts\python.exe



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Step 1 — Download Pretrained Weights (`CT_LiPro_v2.pt`)

Weights are hosted on HuggingFace inside the CT-RATE dataset repository.  
You will need to accept the dataset licence at https://huggingface.co/datasets/ibrahimhamamci/CT-RATE and then log in below.

> **Tip:** Set your HF token once with `huggingface-cli login` instead of pasting it here.

In [16]:
from huggingface_hub import hf_hub_download
import os

# ── Configuration ────────────────────────────────────────────────────────────
HF_DATASET   = "ibrahimhamamci/CT-RATE"
MODELS_DIR   = "./models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Prefer local pretrained weight if already downloaded
LOCAL_LIPRO_WEIGHTS = r"C:\Users\yasiru\Desktop\fyp-3d-ct\models\ct-clip\checkpoints_ctrate_4pathologies\CT_LiPro_v2.pt"

if os.path.isfile(LOCAL_LIPRO_WEIGHTS):
    LIPRO_WEIGHTS = LOCAL_LIPRO_WEIGHTS
    print(f"✅  Using local weights: {LIPRO_WEIGHTS}")
else:
    print("⚠️  Local weights not found. Falling back to HuggingFace download...")

    # ── File map (HuggingFace path → local filename) ─────────────────────────
    WEIGHT_FILES = {
        # PRIMARY — ClassFine model (this pipeline)
        "models/CT-CLIP-Related/CT_LiPro_v2.pt": "CT_LiPro_v2.pt",

        # OPTIONAL — download these too if you want zero-shot / VocabFine
        # "models/CT-CLIP-Related/CT-CLIP_v2.pt": "CT-CLIP_v2.pt",
        # "models/CT-CLIP-Related/CT_VocabFine_v2.pt": "CT_VocabFine_v2.pt",
        # "models/RadBertClassifier.pth": "RadBertClassifier.pth",
    }

    downloaded = {}
    
    for hf_path, local_name in WEIGHT_FILES.items():
        local_path = os.path.join(MODELS_DIR, local_name)
        if os.path.exists(local_path):
            print(f"  [cached]  {local_name}")
        else:
            print(f"  [downloading]  {local_name} ...")
            local_path = hf_hub_download(
                repo_id=HF_DATASET,
                filename=hf_path,
                repo_type="dataset",
                local_dir=MODELS_DIR,
                local_dir_use_symlinks=False,
            )
            # Flatten to MODELS_DIR
            os.rename(local_path, os.path.join(MODELS_DIR, local_name))
        downloaded[local_name] = os.path.join(MODELS_DIR, local_name)

    LIPRO_WEIGHTS = downloaded["CT_LiPro_v2.pt"]
    print(f"\n✅  Weights ready: {LIPRO_WEIGHTS}")

✅  Using local weights: C:\Users\yasiru\Desktop\fyp-3d-ct\models\ct-clip\checkpoints_ctrate_4pathologies\CT_LiPro_v2.pt


---
## Step 2 — Import Libraries & Global Config

In [17]:
import warnings, os, sys, json
warnings.filterwarnings("ignore")

In [18]:
import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from ct_clip import CTCLIP
from transformers import AutoTokenizer, AutoModel
import math
import pickle
from pathlib import Path
import matplotlib.colors as mcolors
import seaborn as sns
from tqdm.auto import tqdm

# Add repo to path
sys.path.insert(0, os.path.abspath("CT-CLIP"))
sys.path.insert(0, os.path.abspath("CT-CLIP/CT_CLIP"))
sys.path.insert(0, os.path.abspath("CT-CLIP/transformer_maskgit"))

# -- Device -------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# -- 18 Pathologies (CT-RATE labels) ------------------------------------------
# Note: We keep all 18 for model loading, but filter to 4 in post-processing
ALL_PATHOLOGIES = [
    "Medical material",
    "Arterial Wall Calcification",
    "Cardiomegaly",
    "Pericardial Effusion",
    "Coronary Artery Wall Calcification",
    "Hiatal Hernia",
    "Lymphadenopathy",
    "Emphysema",
    "Atelectasis",
    "Lung Nodule",
    "Lung Opacity",
    "Pulmonary Fibrotic Sequela",
    "Pleural Effusion",
    "Mosaic Attenuation Pattern",
    "Peribronchial Thickening",
    "Consolidation",
    "Bronchiectasis",
    "Interstitial Lung Disease",
]

# -- Filter to 4 pathologies --------------------------------------------------
PATHOLOGIES_OF_INTEREST = [
    "Lung Nodule",
    "Lung Opacity",
    "Consolidation",
    "Atelectasis",
]

# Get indices of the 4 pathologies in the full list
PATHOLOGY_INDICES = [ALL_PATHOLOGIES.index(p) for p in PATHOLOGIES_OF_INTEREST]

# Use filtered pathologies for display
PATHOLOGIES = PATHOLOGIES_OF_INTEREST
NUM_CLASSES = len(ALL_PATHOLOGIES)  # Keep as 18 for model loading
NUM_CLASSES_FILTERED = len(PATHOLOGIES)  # 4 for display

# -- Volume dimensions expected by CT-ViT -------------------------------------
DEPTH, HEIGHT, WIDTH = 240, 480, 480  # (D, H, W)

# -- HU windowing for chest CT ------------------------------------------------
HU_WINDOW_CENTER = -600
HU_WINDOW_WIDTH = 1500
HU_MIN = HU_WINDOW_CENTER - HU_WINDOW_WIDTH // 2   # -1350
HU_MAX = HU_WINDOW_CENTER + HU_WINDOW_WIDTH // 2   # 150

# -- Classification threshold -------------------------------------------------
THRESHOLD = 0.5

print(f"\nConfig ready: {NUM_CLASSES_FILTERED} pathologies ({PATHOLOGIES}), volume size {DEPTH}x{HEIGHT}x{WIDTH}")

Using device: cpu

Config ready: 4 pathologies (['Lung Nodule', 'Lung Opacity', 'Consolidation', 'Atelectasis']), volume size 240x480x480


---
## Step 3 — Load the CT-CLIP Model + ClassFine Head

**`CT_LiPro_v2.pt`** is the *ClassFine* checkpoint.  
It contains the CT-ViT vision encoder **plus** a linear classification head (`Linear(512 → 18)`) trained with binary cross-entropy on the CT-RATE abnormality labels.  
The text encoder is **not used** for this variant — making inference ~3× faster.

In [34]:
print("=" * 80)
print("DEBUG: Checkpoint state_dict Structure")
print("=" * 80)

# Load checkpoint
ckpt_raw = torch.load(LIPRO_WEIGHTS, map_location="cpu")
ckpt_state = ckpt_raw.get("model_state_dict", ckpt_raw.get("state_dict", ckpt_raw))

print(f"\nCheckpoint has 'trained_model' prefix?")
sample_keys = list(ckpt_state.keys())[:5]
for key in sample_keys:
    print(f"  {key}")

# Check if trained_model prefix is present
has_trained_prefix = any(k.startswith("trained_model.") for k in ckpt_state.keys())
print(f"\nHas 'trained_model.' prefix: {has_trained_prefix}")

# The classifier is the ClassFine head, find it
classifier_keys = [k for k in ckpt_state.keys() if 'classifier' in k.lower()]
print(f"\nClassifier-related keys: {classifier_keys}")

# Count state_dict keys that would map to the wrapper vs CTCLIP model
wrapper_keys = ['classifier']
for key in ckpt_state.keys():
    if 'classifier' in key or 'clip_model' not in key:
        if 'trained_model' in key:
            print(f"  found trained model key: {key}")
            break

print("\n" + "=" * 80)


DEBUG: Checkpoint state_dict Structure

Checkpoint has 'trained_model' prefix?
  trained_model.temperature
  trained_model.text_transformer.embeddings.position_ids
  trained_model.text_transformer.embeddings.word_embeddings.weight
  trained_model.text_transformer.embeddings.position_embeddings.weight
  trained_model.text_transformer.embeddings.token_type_embeddings.weight

Has 'trained_model.' prefix: True

Classifier-related keys: ['classifier.weight', 'classifier.bias']
  found trained model key: trained_model.temperature



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from ct_clip import CTCLIP
from transformers import AutoTokenizer, AutoModel
import math
import pickle

# ─────────────────────────────────────────────────────────────────────────────
# 3A. Load pretrained CTCLIP model and classifier head
# ─────────────────────────────────────────────────────────────────────────────
print("Loading CT_LiPro_v2.pt checkpoint …")

# Resolve the checkpoint path locally so this cell can run on its own.
if 'LIPRO_WEIGHTS' not in globals() or not LIPRO_WEIGHTS:
    candidate = Path(r"C:\Users\yasiru\Desktop\fyp-3d-ct\models\ct-clip\checkpoints_ctrate_4pathologies\CT_LiPro_v2.pt")
    if candidate.is_file():
        LIPRO_WEIGHTS = str(candidate)
        print(f"  Using local weights: {LIPRO_WEIGHTS}")
    elif globals().get('LOCAL_LIPRO_WEIGHTS') and Path(LOCAL_LIPRO_WEIGHTS).is_file():
        LIPRO_WEIGHTS = LOCAL_LIPRO_WEIGHTS
        print(f"  Using configured weights: {LIPRO_WEIGHTS}")
    else:
        raise FileNotFoundError(
            'LIPRO_WEIGHTS is not defined. Run Step 1 first or point the notebook to the CT_LiPro_v2.pt file.'
        )

# Load the checkpoint
ckpt = torch.load(LIPRO_WEIGHTS, map_location="cpu")
# The checkpoint contains both trained_model (CTCLIP) and classifier
print(f"  Checkpoint keys: {list(state.keys())[:3]}...")
print(f"  Total state keys: {len(state)}")

# Instantiate the text encoder
TEXT_ENCODER_NAME = "microsoft/BiomedVLP-CXR-BERT-specialized"
tokenizer = AutoTokenizer.from_pretrained(
    TEXT_ENCODER_NAME,
    trust_remote_code=True
)
print(f"\n  Text encoder loaded: {TEXT_ENCODER_NAME}")

# Determine dim_image from the checkpoint's to_visual_latent layer
# Shape is (dim_latent, dim_image)
for key in state.keys():
    if 'to_visual_latent.weight' in key:
        to_visual_latent_shape = state[key].shape
        dim_latent, dim_image = to_visual_latent_shape
        print(f"  Found to_visual_latent: {to_visual_latent_shape}")
        print(f"    → dim_latent={dim_latent}, dim_image={dim_image}")
        break

# Create CTCLIP with correct dimensions
model = CTCLIP(
    image_encoder   = None,
    text_encoder    = TEXT_ENCODER_NAME,
    dim_image       = dim_image,
    dim_text        = 768,
    dim_latent      = dim_latent,
    extra_latent_projection = False,
)

# ─────────────────────────────────────────────────────────────────────────────
# 3B. ClassFine wrapper
# ─────────────────────────────────────────────────────────────────────────────
class CTCLIPClassFine(nn.Module):
    """
    Wraps the CT-CLIP model to extract image latents and classify with a linear head.
    """

    def __init__(self, clip_model: CTCLIP, num_classes: int = 18):
        super().__init__()
        self.clip_model = clip_model
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, images: torch.Tensor):
        batch_size = images.shape[0]
        dummy_text = tokenizer(
            [""] * batch_size,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77,
        ).to(images.device)
        
        _, image_latents, _ = self.clip_model(
            text=dummy_text,
            image=images,
            device=images.device,
            return_latents=True,
        )
        
        logits = self.classifier(image_latents)
        probs = torch.sigmoid(logits)
        return logits, probs


# ─────────────────────────────────────────────────────────────────────────────
# 3C. Load checkpoint weights
# ─────────────────────────────────────────────────────────────────────────────
clf_model = CTCLIPClassFine(model, num_classes=NUM_CLASSES)

# Remap keys: trained_model.XXX → clip_model.XXX
state_remapped = {
    k.replace("module.", "")
     .replace("_orig_mod.", "")
     .replace("trained_model.", "clip_model."): v
    for k, v in state.items()
}

missing, unexpected = clf_model.load_state_dict(state_remapped, strict=False)
print(f"\n  Missing keys  : {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")

clf_model = clf_model.to(DEVICE).eval()

total_params = sum(p.numel() for p in clf_model.parameters())
print(f"\n✅  Model loaded — {total_params:,} parameters")

c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:261: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:391: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:261: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:391: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


Loading CT_LiPro_v2.pt checkpoint …
  Checkpoint keys: ['trained_model.temperature', 'trained_model.text_transformer.embeddings.position_ids', 'trained_model.text_transformer.embeddings.word_embeddings.weight']...
  Total state keys: 364


c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:261: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:391: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


Loading CT_LiPro_v2.pt checkpoint …
  Checkpoint keys: ['trained_model.temperature', 'trained_model.text_transformer.embeddings.position_ids', 'trained_model.text_transformer.embeddings.word_embeddings.weight']...
  Total state keys: 364


...

c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:261: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
c:\Users\yasiru\Desktop\fyp-3d-ct\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:391: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


Loading CT_LiPro_v2.pt checkpoint …
  Checkpoint keys: ['trained_model.temperature', 'trained_model.text_transformer.embeddings.position_ids', 'trained_model.text_transformer.embeddings.word_embeddings.weight']...
  Total state keys: 364


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


  Text encoder loaded: microsoft/BiomedVLP-CXR-BERT-specialized


NameError: name 'NUM_CLASSES' is not defined

---
## Step 4 — CT Volume Preprocessing

CT-CLIP expects **non-contrast chest CT volumes** preprocessed as follows:

| Step | Detail |
|------|--------|
| HU Windowing | centre −600, width 1500 → clamp to [−1350, 150] |
| Normalise | linear map to [0, 1] |
| Resample | isotropic voxel spacing via SimpleITK |
| Resize | tri-linear interpolation → (D=240, H=480, W=480) |
| Channel | add channel dim → (1, 240, 480, 480) |
| Batch | add batch dim → (1, 1, 240, 480, 480) |

In [3]:
import SimpleITK as sitk
from scipy.ndimage import zoom


def load_nifti(path: str) -> tuple:
    """
    Load a NIfTI file and return (volume_np, spacing).
    Works with both .nii and .nii.gz.
    """
    itk_img = sitk.ReadImage(path)
    spacing = itk_img.GetSpacing()          # (x, y, z) mm / voxel
    volume  = sitk.GetArrayFromImage(itk_img)  # (z, y, x) i.e. (D, H, W)
    return volume.astype(np.float32), spacing


def hu_window(volume: np.ndarray,
              center: float = HU_WINDOW_CENTER,
              width : float = HU_WINDOW_WIDTH) -> np.ndarray:
    """Clip to the HU window and normalise to [0, 1]."""
    lo = center - width / 2.0
    hi = center + width / 2.0
    v  = np.clip(volume, lo, hi)
    return (v - lo) / (hi - lo)    # → [0, 1]


def resample_isotropic(volume: np.ndarray,
                       spacing: tuple,
                       target_spacing: float = 1.0) -> np.ndarray:
    """
    Resample volume to isotropic voxel spacing.
    spacing = (sx, sy, sz) in mm (SimpleITK order: x, y, z)
    volume  shape: (z, y, x)
    """
    sz, sy, sx = spacing[2], spacing[1], spacing[0]
    factors = (sz / target_spacing,
               sy / target_spacing,
               sx / target_spacing)
    return zoom(volume, factors, order=1)   # bilinear


def resize_volume(volume: np.ndarray,
                  target_shape: tuple = (DEPTH, HEIGHT, WIDTH)) -> np.ndarray:
    """Resize (D, H, W) volume to target_shape with bilinear zoom."""
    d, h, w = volume.shape
    td, th, tw = target_shape
    factors = (td / d, th / h, tw / w)
    return zoom(volume, factors, order=1)


def preprocess_ct(
    path: str,
    resample: bool = True,
    target_spacing: float = 1.0,
    target_shape : tuple = (DEPTH, HEIGHT, WIDTH),
    device: torch.device = DEVICE,
) -> torch.Tensor:
    """
    Full preprocessing pipeline for a single NIfTI CT volume.

    Returns:
        tensor : (1, 1, D, H, W)  float32 on `device`, values ∈ [0, 1]
    """
    print(f"  Loading  {os.path.basename(path)} …")
    vol, spacing = load_nifti(path)
    print(f"  Raw shape : {vol.shape}, spacing: ({spacing[0]:.2f}, {spacing[1]:.2f}, {spacing[2]:.2f}) mm")

    # 1. HU windowing + normalisation
    vol = hu_window(vol)

    # 2. Isotropic resampling (optional but recommended)
    if resample:
        vol = resample_isotropic(vol, spacing, target_spacing)
        print(f"  After resample : {vol.shape}")

    # 3. Resize to model input size
    vol = resize_volume(vol, target_shape)
    print(f"  After resize   : {vol.shape}  → {target_shape}")

    # 4. Convert to tensor and add batch + channel dims
    tensor = torch.from_numpy(vol).float().unsqueeze(0).unsqueeze(0)  # (1, 1, D, H, W)
    return tensor.to(device)


# ── Preview helper ────────────────────────────────────────────────────────────
def preview_volume(tensor: torch.Tensor, title: str = "CT Volume") -> None:
    """
    Show axial, coronal, and sagittal mid-slices of the preprocessed volume.
    tensor: (1, 1, D, H, W)
    """
    vol = tensor[0, 0].cpu().numpy()   # (D, H, W)
    d, h, w = vol.shape

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    axes[0].imshow(vol[d // 2], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(f"Axial  (slice {d // 2}/{d})")

    axes[1].imshow(vol[:, h // 2, :], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Coronal  (slice {h // 2}/{h})")

    axes[2].imshow(vol[:, :, w // 2], cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"Sagittal  (slice {w // 2}/{w})")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("✅  Preprocessing functions defined.")

NameError: name 'HU_WINDOW_CENTER' is not defined

---
## Step 5 — Load & Preview a CT Volume

Replace `CT_VOLUME_PATH` with the path to your `.nii` or `.nii.gz` file.  
If you don't have one yet, the cell below generates a **synthetic phantom** so you can test the whole pipeline end-to-end.

In [4]:
# ── 1) Point to your real NIfTI file ─────────────────────────────────────────
CT_VOLUME_PATH = r"C:\Users\yasiru\Desktop\fyp-3d-ct\data\ct-rate\data_Volumes\train_fixed\train_9\train_9_a\train_9_a_1.nii.gz"


# ── 2) Fallback: synthetic phantom (Gaussian blobs mimicking lungs) ───────────
def make_synthetic_ct(
    save_path: str = "/tmp/synthetic_ct.nii.gz",
    shape: tuple = (120, 240, 240),    # smaller for speed
    spacing: tuple = (2.0, 2.0, 2.0),
) -> str:
    """Create a synthetic chest-phantom NIfTI for pipeline testing."""
    from skimage.draw import disk

    d, h, w = shape
    vol = np.full(shape, -900.0, dtype=np.float32)   # lung air approx -900 HU

    # soft tissue shell (chest wall)
    for z in range(d):
        rr, cc = disk((h // 2, w // 2), min(h, w) // 2 - 5, shape=(h, w))
        vol[z, rr, cc] = -50.0   # soft tissue approx -50 HU

    # lung cavities
    for z in range(d):
        for cx, cy, r in [(h // 2, w // 3, h // 4), (h // 2, 2 * w // 3, h // 4)]:
            rr, cc = disk((cx, cy), r, shape=(h, w))
            vol[z, rr, cc] = -800.0   # lung parenchyma approx -800 HU

    # trachea
    for z in range(d):
        rr, cc = disk((h // 5, w // 2), 8, shape=(h, w))
        vol[z, rr, cc] = -950.0

    # heart (central soft tissue)
    for z in range(d // 3, 2 * d // 3):
        rr, cc = disk((h // 2, w // 2), h // 8, shape=(h, w))
        vol[z, rr, cc] = 60.0   # blood / myocardium

    # add noise
    vol += np.random.normal(0, 20, vol.shape).astype(np.float32)

    # save
    img = nib.Nifti1Image(vol.transpose(2, 1, 0), np.diag([*spacing, 1.0]))
    nib.save(img, save_path)
    print(f"  Synthetic phantom saved to {save_path}")
    return save_path


if not os.path.isfile(CT_VOLUME_PATH):
    print("Real NIfTI not found, generating synthetic phantom for testing ...")
    CT_VOLUME_PATH = make_synthetic_ct()

# ── Preprocess ───────────────────────────────────────────────────────────────
ct_tensor = preprocess_ct(CT_VOLUME_PATH)
print(f"\nFinal tensor shape: {ct_tensor.shape}")
print(f"Value range       : [{ct_tensor.min():.3f}, {ct_tensor.max():.3f}]")

# ── Visualise mid-slices ─────────────────────────────────────────────────────
preview_volume(ct_tensor, title=f"Preprocessed CT - {os.path.basename(CT_VOLUME_PATH)}")

NameError: name 'os' is not defined

---
## Step 6 — Run Inference with `CT_LiPro_v2.pt`

The ClassFine model passes the CT volume through the **CT-ViT encoder** then through the **linear classification head**, yielding 18 independent sigmoid probabilities — one per pathology.

In [5]:
import time

@torch.inference_mode()
def run_classfine_inference(
    model   : CTCLIPClassFine,
    ct_vol  : torch.Tensor,
    threshold: float = THRESHOLD,
) -> dict:
    """
    Run CT_LiPro_v2 (ClassFine) inference on a single preprocessed CT tensor.

    Args:
        model    : CTCLIPClassFine loaded with CT_LiPro_v2.pt
        ct_vol   : (1, 1, D, H, W)  float32 on DEVICE
        threshold: decision boundary for binary labels

    Returns:
        dict with keys:
            'probabilities' : np.ndarray shape (4,) — filtered to 4 pathologies
            'predictions'   : np.ndarray shape (4,)  bool
            'positive_findings' : list[str]
            'inference_time_s'  : float
    """
    model.eval()

    # Mixed precision for speed if GPU is available
    autocast_ctx = (torch.cuda.amp.autocast()
                    if DEVICE.type == "cuda" else torch.no_grad())

    t0 = time.perf_counter()
    with autocast_ctx:
        logits, probs = model(ct_vol)       # (1, 18)
    t1 = time.perf_counter()

    probs_np = probs[0].cpu().float().numpy()       # (18,)
    
    # Filter to only the 4 pathologies of interest
    probs_filtered = probs_np[PATHOLOGY_INDICES]    # (4,)
    preds_filtered = probs_filtered >= threshold    # (4,) bool

    positive = [PATHOLOGIES[i] for i in range(NUM_CLASSES_FILTERED) if preds_filtered[i]]

    return {
        "probabilities"    : probs_filtered,
        "predictions"      : preds_filtered,
        "positive_findings": positive,
        "inference_time_s" : t1 - t0,
    }


# ── Run ──────────────────────────────────────────────────────────────────────
print("Running inference …")
results = run_classfine_inference(clf_model, ct_tensor)

print(f"\n⏱  Inference time : {results['inference_time_s']:.3f} s")
print(f"\n{'Pathology':<40} {'Probability':>12}  {'Detected':>9}")
print("-" * 65)
for name, prob, pred in zip(
    PATHOLOGIES, results["probabilities"], results["predictions"]
):
    flag = "✅  YES" if pred else "—"
    bar  = "█" * int(prob * 20)
    print(f"{name:<40} {prob:>10.3f}  {flag:>9}  {bar}")

print(f"\n🩻  Positive findings ({len(results['positive_findings'])}/{NUM_CLASSES_FILTERED}):")
for f in results["positive_findings"]:
    print(f"   • {f}")
if not results["positive_findings"]:
    print("   (none above threshold)")


NameError: name 'THRESHOLD' is not defined

---
## Step 7 — Visualise Results

In [ ]:
# ── 7A. Horizontal Bar Chart ──────────────────────────────────────────────────
probs = results["probabilities"]
preds = results["predictions"]

sort_idx = np.argsort(probs)[::-1]
sorted_names  = [PATHOLOGIES[i] for i in sort_idx]
sorted_probs  = probs[sort_idx]
sorted_preds  = preds[sort_idx]

colors = ["#e74c3c" if p else "#3498db" for p in sorted_preds]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(sorted_names, sorted_probs, color=colors, edgecolor="white", height=0.65)
ax.axvline(THRESHOLD, color="black", linestyle="--", linewidth=1.2, label=f"Threshold = {THRESHOLD}")

for bar, prob in zip(bars, sorted_probs):
    ax.text(min(prob + 0.01, 0.97), bar.get_y() + bar.get_height() / 2,
            f"{prob:.3f}", va="center", fontsize=8.5)

ax.set_xlim(0, 1.05)
ax.set_xlabel("Probability", fontsize=11)
ax.set_title("CT-CLIP (CT_LiPro_v2) — 4 Pathologies (Lung Nodule, Opacity, Consolidation, Atelectasis)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)

# Custom legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor="#e74c3c", label="Positive (≥ threshold)"),
              Patch(facecolor="#3498db", label="Negative (< threshold)")]
ax.legend(handles=legend_els + ax.lines, fontsize=9, loc="lower right")

plt.tight_layout()
plt.savefig("/tmp/ctclip_bar.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 7B. Radar / Spider Chart ──────────────────────────────────────────────────
import matplotlib.patches as mpatches

N = NUM_CLASSES_FILTERED
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values = probs.tolist()

# Close the polygon
angles  += angles[:1]
values  += values[:1]
labels   = PATHOLOGIES + [PATHOLOGIES[0]]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

ax.plot(angles, values, "o-", linewidth=2, color="#2980b9")
ax.fill(angles, values, alpha=0.22, color="#2980b9")

# Threshold ring
thresh_ring = [THRESHOLD] * (N + 1)
ax.plot(angles, thresh_ring, linestyle="--", color="#e74c3c",
        linewidth=1.2, label=f"Threshold = {THRESHOLD}")

ax.set_thetagrids(np.degrees(angles[:-1]), PATHOLOGIES, fontsize=8)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=7)
ax.set_title("CT-CLIP (CT_LiPro_v2) — Pathology Radar Chart (4 of Interest)",
             y=1.08, fontsize=13, fontweight="bold")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig("/tmp/ctclip_radar.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 7C. Summary table ─────────────────────────────────────────────────────────
df = pd.DataFrame({
    "Pathology"  : PATHOLOGIES,
    "Probability": np.round(probs, 4),
    "Detected"   : ["Yes" if p else "No" for p in preds],
}).sort_values("Probability", ascending=False).reset_index(drop=True)

def colour_row(row):
    if row["Detected"] == "Yes":
        return ["background-color: #fdecea"] * len(row)
    return [""] * len(row)

styled_df = df.style.apply(colour_row, axis=1).bar(
    subset=["Probability"], vmin=0, vmax=1, color="#74b9ff"
)
print("\n📊 Summary Table (4 Pathologies of Interest):")
print(df.to_string(index=False))


---
## Step 8 — Batch Inference Over a Folder

For evaluating multiple CT volumes. Results are saved to `ctclip_results.csv`.

In [ ]:
class CTVolumeDataset(Dataset):
    """Simple Dataset for a folder of NIfTI files."""

    def __init__(self, folder: str, extensions=(".nii", ".nii.gz")):
        self.files = sorted([
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if any(f.endswith(ext) for ext in extensions)
        ])
        if not self.files:
            raise FileNotFoundError(f"No NIfTI files found in {folder}")
        print(f"Found {len(self.files)} CT volume(s) in {folder}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        try:
            tensor = preprocess_ct(path, device=torch.device("cpu"))
            return tensor[0], os.path.basename(path), True
        except Exception as e:
            print(f"  ⚠  Error loading {path}: {e}")
            # Return zero tensor as placeholder
            return torch.zeros(1, DEPTH, HEIGHT, WIDTH), os.path.basename(path), False


@torch.inference_mode()
def batch_inference(
    model    : CTCLIPClassFine,
    folder   : str,
    threshold: float = THRESHOLD,
    output_csv: str = "ctclip_results.csv",
) -> pd.DataFrame:
    """
    Run CT_LiPro_v2 inference over all NIfTI files in `folder`.
    Saves results to `output_csv` and returns a DataFrame.
    Only reports the 4 pathologies of interest.
    """
    dataset = CTVolumeDataset(folder)
    loader  = DataLoader(dataset, batch_size=1, num_workers=2, pin_memory=True)

    rows = []
    model.eval()

    for vol, fname, ok in tqdm(loader, desc="Inferring"):
        fname = fname[0]
        if not ok[0]:
            rows.append({"file": fname, **{p: np.nan for p in PATHOLOGIES}})
            continue

        vol = vol.to(DEVICE)               # (1, 1, D, H, W)
        _, probs = model(vol)
        probs_np = probs[0].cpu().float().numpy()    # (18,)
        
        # Filter to 4 pathologies of interest
        probs_filtered = probs_np[PATHOLOGY_INDICES]

        row = {"file": fname}
        for name, prob in zip(PATHOLOGIES, probs_filtered):
            row[name] = round(float(prob), 4)
        row["n_positive"] = int((probs_filtered >= threshold).sum())
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"\n✅  Results saved to {output_csv}")
    return df


# ── To run batch inference, set a folder path and uncomment below: ────────────
# CT_FOLDER = "/path/to/ct/volumes/"
# df_results = batch_inference(clf_model, CT_FOLDER)
# df_results.head()

print("✅  Batch inference functions defined. Uncomment the last lines to run.")


---
## Step 9 — (Optional) Zero-Shot Comparison with `CT-CLIP_v2.pt`

Zero-shot uses **both encoders** and prompt templates `"{Abnormality} is present."` / `"{Abnormality} is not present."`.  
No classification head is required — the contrastive similarity drives prediction.  
Download `CT-CLIP_v2.pt` first (uncomment in Step 1).

In [ ]:
ZEROSHOT_WEIGHTS = "./models/CT-CLIP_v2.pt"    # set path after downloading


@torch.inference_mode()
def run_zeroshot_inference(
    clip_model : CTCLIP,
    tokenizer,
    ct_vol     : torch.Tensor,
    pathologies: list = PATHOLOGIES,
    threshold  : float = THRESHOLD,
    max_length : int = 77,
) -> dict:
    """
    Zero-shot multi-abnormality detection via prompt similarity.

    For each pathology two prompts are encoded:
        pos: "{Abnormality} is present."
        neg: "{Abnormality} is not present."

    Score = softmax([sim(img, pos), sim(img, neg)])[0]  → P(positive)
    """
    clip_model.eval()

    autocast_ctx = (torch.cuda.amp.autocast()
                    if DEVICE.type == "cuda" else torch.no_grad())

    t0 = time.perf_counter()
    with autocast_ctx:
        # Encode image once
        img_emb = clip_model.encode_image(ct_vol)   # (1, 512)
        img_emb = F.normalize(img_emb, dim=-1)

        probs_all = []
        for abnorm in pathologies:
            pos_prompt = f"{abnorm} is present."
            neg_prompt = f"{abnorm} is not present."

            def encode_text(text):
                tokens = tokenizer(
                    text, return_tensors="pt",
                    max_length=max_length,
                    padding="max_length",
                    truncation=True,
                ).to(DEVICE)
                txt_emb = clip_model.encode_text(**tokens)  # (1, 512)
                return F.normalize(txt_emb, dim=-1)

            pos_emb = encode_text(pos_prompt)
            neg_emb = encode_text(neg_prompt)

            # Cosine similarities
            sim_pos = (img_emb * pos_emb).sum(dim=-1)  # scalar
            sim_neg = (img_emb * neg_emb).sum(dim=-1)  # scalar

            logits = torch.stack([sim_pos, sim_neg], dim=-1) * clip_model.temperature.exp()
            p_pos  = torch.softmax(logits, dim=-1)[0, 0].item()
            probs_all.append(p_pos)

    t1 = time.perf_counter()
    probs_np = np.array(probs_all)
    preds_np = probs_np >= threshold

    return {
        "probabilities"    : probs_np,
        "predictions"      : preds_np,
        "positive_findings": [pathologies[i] for i in range(len(pathologies)) if preds_np[i]],
        "inference_time_s" : t1 - t0,
    }


# ── Load zero-shot model and compare (uncomment when CT-CLIP_v2.pt is ready) ─
# if os.path.isfile(ZEROSHOT_WEIGHTS):
#     zs_model = CTCLIP(
#         image_encoder=None,
#         text_encoder=TEXT_ENCODER_NAME,
#         dim_image=512, dim_text=768, dim_latent=512,
#     )
#     ckpt_zs = torch.load(ZEROSHOT_WEIGHTS, map_location="cpu")
#     state_zs = ckpt_zs.get("model_state_dict", ckpt_zs)
#     state_zs = {k.replace("module.","").replace("_orig_mod.",""): v for k, v in state_zs.items()}
#     zs_model.load_state_dict(state_zs, strict=False)
#     zs_model = zs_model.to(DEVICE).eval()
#
#     zs_results = run_zeroshot_inference(zs_model, tokenizer, ct_tensor)
#     print(f"Zero-shot inference time: {zs_results['inference_time_s']:.2f} s")
#     print("Positive findings:", zs_results["positive_findings"])
# else:
#     print("⚠  CT-CLIP_v2.pt not found — skip zero-shot comparison.")

print("✅  Zero-shot comparison code ready. Uncomment to run after downloading CT-CLIP_v2.pt.")

---
## Step 10 — Embedding Extraction & Case Retrieval

Extract visual embeddings from any number of CT volumes and perform **image-to-image retrieval** by cosine similarity — mirroring the paper's MAP@K experiments.

In [ ]:
@torch.inference_mode()
def extract_embedding(
    model  : CTCLIPClassFine,
    ct_vol : torch.Tensor,
) -> np.ndarray:
    """Return the L2-normalised 512-dim visual embedding for one volume."""
    model.eval()
    emb = model.encode_image(ct_vol)    # (1, 512)
    return emb[0].cpu().float().numpy()


def cosine_retrieval(
    query_emb  : np.ndarray,
    gallery_embs: np.ndarray,
    gallery_ids : list,
    top_k      : int = 5,
) -> list:
    """
    Retrieve the top-K most similar cases from a gallery.

    Args:
        query_emb   : (512,)    L2-normalised query embedding
        gallery_embs: (N, 512)  L2-normalised gallery embeddings
        gallery_ids : list of N case identifiers
        top_k       : number of results to return

    Returns:
        list of (case_id, similarity_score) tuples, sorted descending
    """
    sims = gallery_embs @ query_emb   # (N,)
    idx  = np.argsort(sims)[::-1][:top_k]
    return [(gallery_ids[i], float(sims[i])) for i in idx]


# ── Demo: single-volume self-similarity ──────────────────────────────────────
query_emb = extract_embedding(clf_model, ct_tensor)
print(f"Embedding shape : {query_emb.shape}")
print(f"Embedding norm  : {np.linalg.norm(query_emb):.4f}  (should be ≈ 1.0)")

# Self-similarity (trivially = 1.0)
gallery = np.stack([query_emb])   # gallery with one entry
hits = cosine_retrieval(query_emb, gallery, ["query_case"], top_k=1)
print(f"Self-similarity : {hits[0][1]:.4f}")

print("\n✅  Embedding extraction & retrieval ready.")

---
## Step 11 — Evaluation Against Ground-Truth Labels

If you have ground-truth binary labels for 18 pathologies, compute AUROC, F1, and accuracy per pathology — replicating the paper's benchmark setup.

In [ ]:
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    precision_score, recall_score, classification_report,
    roc_curve, auc,
)


@torch.inference_mode()
def evaluate(
    model       : CTCLIPClassFine,
    nifti_paths : list,
    labels_array: np.ndarray,    # (N, 18)  binary ground truth
    threshold   : float = THRESHOLD,
    output_csv  : str = "ctclip_eval.csv",
) -> pd.DataFrame:
    """
    Evaluate CT_LiPro_v2 on a labelled test set.

    Args:
        nifti_paths  : list of N NIfTI file paths
        labels_array : (N, 18) binary GT labels in PATHOLOGIES order

    Returns:
        DataFrame with per-pathology AUROC, F1, Accuracy, Precision, Recall
    """
    assert labels_array.shape == (len(nifti_paths), NUM_CLASSES), \
        f"labels_array must be ({len(nifti_paths)}, {NUM_CLASSES})"

    model.eval()
    all_probs = []

    for path in tqdm(nifti_paths, desc="Evaluating"):
        tensor = preprocess_ct(path)
        _, probs = model(tensor)
        all_probs.append(probs[0].cpu().float().numpy())

    all_probs = np.stack(all_probs)      # (N, 18)
    all_preds = (all_probs >= threshold).astype(int)

    metrics = []
    for i, name in enumerate(PATHOLOGIES):
        gt  = labels_array[:, i]
        pr  = all_probs[:, i]
        pred = all_preds[:, i]
        try:
            auroc = roc_auc_score(gt, pr)
        except ValueError:
            auroc = float("nan")
        metrics.append({
            "Pathology" : name,
            "AUROC"     : round(auroc, 4),
            "F1"        : round(f1_score(gt, pred, zero_division=0), 4),
            "Accuracy"  : round(accuracy_score(gt, pred), 4),
            "Precision" : round(precision_score(gt, pred, zero_division=0), 4),
            "Recall"    : round(recall_score(gt, pred, zero_division=0), 4),
        })

    df = pd.DataFrame(metrics)
    mean_row = df.mean(numeric_only=True).to_dict()
    mean_row["Pathology"] = "MEAN"
    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)

    df.to_csv(output_csv, index=False)
    print(f"\n✅  Evaluation saved to {output_csv}")
    return df


# ── Plot AUROC heatmap from a pre-computed results DataFrame ──────────────────
def plot_auroc_heatmap(df: pd.DataFrame) -> None:
    """Seaborn heatmap of per-pathology AUROC values."""
    subset = df[df["Pathology"] != "MEAN"][["Pathology", "AUROC"]].set_index("Pathology")
    fig, ax = plt.subplots(figsize=(3, 8))
    sns.heatmap(
        subset, annot=True, fmt=".3f", cmap="RdYlGn",
        vmin=0.5, vmax=1.0, linewidths=0.5, ax=ax
    )
    ax.set_title("AUROC per Pathology\n(CT_LiPro_v2)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


# ── Usage (uncomment with real data) ─────────────────────────────────────────
# test_paths  = ["case001.nii.gz", "case002.nii.gz", ...]     # N paths
# test_labels = np.load("test_labels.npy")                      # (N, 18)
# eval_df = evaluate(clf_model, test_paths, test_labels)
# print(eval_df.tail(3))     # show last rows incl. MEAN
# plot_auroc_heatmap(eval_df)

print("✅  Evaluation scaffold ready.")

---
## Step 12 — Export Report as JSON

In [ ]:
import json
from datetime import datetime

report = {
    "model"           : "CT_LiPro_v2.pt (CT-CLIP ClassFine)",
    "volume"          : os.path.basename(CT_VOLUME_PATH),
    "inference_time_s": round(results["inference_time_s"], 4),
    "threshold"       : THRESHOLD,
    "timestamp"       : datetime.now().isoformat(),
    "findings": [
        {
            "pathology"  : name,
            "probability": round(float(prob), 4),
            "positive"   : bool(pred),
        }
        for name, prob, pred in zip(
            PATHOLOGIES, results["probabilities"], results["predictions"]
        )
    ],
    "positive_findings" : results["positive_findings"],
    "n_positive"        : len(results["positive_findings"]),
}

REPORT_PATH = "ctclip_report.json"
with open(REPORT_PATH, "w") as fh:
    json.dump(report, fh, indent=2)

print(f"✅  Report saved to {REPORT_PATH}")
print(json.dumps(report, indent=2))

---
## Quick Reference

### Model Weights (HuggingFace)

| Checkpoint | HF Path | Variant |
|---|---|---|
| `CT_LiPro_v2.pt` | `models/CT-CLIP-Related/CT_LiPro_v2.pt` | ClassFine (this notebook) |
| `CT-CLIP_v2.pt` | `models/CT-CLIP-Related/CT-CLIP_v2.pt` | Zero-shot |
| `CT_VocabFine_v2.pt` | `models/CT-CLIP-Related/CT_VocabFine_v2.pt` | VocabFine |
| `RadBertClassifier.pth` | `models/RadBertClassifier.pth` | Text label extractor |

### Inference Modes

| Mode | Code | Speed | Use case |
|---|---|---|---|
| **ClassFine** (CT_LiPro_v2) | Step 6 | ~0.5 s | Fast 18-class detection |
| **Zero-shot** (CT-CLIP_v2) | Step 9 | ~1.5 s | Open-vocabulary, retrieval |
| **VocabFine** | Step 9 | ~1.5 s | Best accuracy + open vocab |

### 18 Pathologies

Medical material · Arterial Wall Calcification · Cardiomegaly · Pericardial Effusion · Coronary Artery Wall Calcification · Hiatal Hernia · Lymphadenopathy · Emphysema · Atelectasis · Lung Nodule · Lung Opacity · Pulmonary Fibrotic Sequela · Pleural Effusion · Mosaic Attenuation Pattern · Peribronchial Thickening · Consolidation · Bronchiectasis · Interstitial Lung Disease

### Citation

```bibtex
@article{hamamci2024ctrate,
  title   = {Generalist Foundation Models from a Multimodal Dataset for 3D Computed Tomography},
  author  = {Hamamci, Ibrahim Ethem and Er, Sezgin and others},
  journal = {arXiv:2403.17834},
  year    = {2024}
}
```